### RecipeNLG (cooking recipes dataset)

1. Read the dataset

In [ ]:
import pandas as pd
# Count occurrences of each ingredient
from collections import Counter

In [ ]:

dataset_path = "D:\LLM\Recipies Finetuning\dataset\RecipeNLG\RecipeNLG_dataset.csv"
df = pd.read_csv(dataset_path)
df = df.rename(columns={'Unnamed: 0': 'recipe_id'})
df.head()

Observation ; Few  of the titles are havng more than one recipe. They have different ingredients and directions. How to handle this?

### Length of ingredients

In [ ]:
import ast

def beautify_print_ingredients(ingredients_str):
    # Convert string representation of list to actual list
    try:
        ingredients_list = ast.literal_eval(ingredients_str) if isinstance(ingredients_str, str) else ingredients_str
        # Pretty print the list with each ingredient on a new line
        print("Ingredients:")
        for i, ingredient in enumerate(ingredients_list, 1):
            print(f"{i}. {ingredient}")
    except Exception as e:
        print("Error formatting ingredients:", e)
        print("Original string:")
        print(ingredients_str)

# Cleaning
Replacing the tsp and c etc...

In [ ]:
# 1) Unit variations / canonical map
# -------------------------
UNIT_VARIATIONS = {
    "cup": ["c", "c.", "cup", "cups"],
    "teaspoon": ["tsp", "tsp.", "tsps", "tsps.", "teaspoon", "teaspoons", "t"],
    "tablespoon": ["tbsp", "tbsp.", "tbs", "tbs.", "tablespoon", "tablespoons", "tbl", "T", "T."],
    "ounce": ["oz", "oz.", "ounce", "ounces"],
    "pound": ["lb", "lb.", "lbs", "lbs.", "pound", "pounds"],
    "gram": ["g", "g.", "gram", "grams"],
    "kilogram": ["kg", "kg.", "kilogram", "kilograms"],
    "milliliter": ["ml", "ml.", "milliliter", "milliliters"],
    "liter": ["l", "l.", "liter", "liters"],
    "pinch": ["pinch", "pinches"],
    "dash": ["dash", "dashes"],
    "package": ["pkg", "pkg.", "package", "packages", "pack"],
    "slice": ["slice", "slices"],
    "clove": ["clove", "cloves"],
    "stick": ["stick", "sticks"],
    "quart": ["qt", "qt.", "quart", "quarts"],
    "gallon": ["gal", "gal.", "gallon", "gallons"],
    # extend as needed...
}

Build a reverse lookup dictionary

In [ ]:
import re
# build reverse map: variation -> canonical
UNIT_MAP = {}
for can, vars in UNIT_VARIATIONS.items():
    for v in vars:
        UNIT_MAP[v.lower().rstrip(".")] = can

# Compile regex pattern for units (strip trailing dot in keys)
UNIT_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in sorted(set(k.rstrip(".") for k in UNIT_MAP.keys()), key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

Universal regex-based unit normalizer

In [ ]:
def normalize_units_advanced(text):
    def replace_unit(match):
        raw = match.group(1).lower().replace(".", "")
        return UNIT_MAP.get(raw, raw)

    return UNIT_PATTERN.sub(replace_unit, text)


In [ ]:
# -------------------------
# 2) Noise words and patterns
# -------------------------
NOISE_WORDS = [
    r"\(optional\)",
    r"\boptional\b",
    r"\bto taste\b",
    r"\bdivided\b",
    r"\bas needed\b",
    r"\b(or more)\b",
    r"\bplus extra\b",
    r"\bfor garnish\b",
    r"\bpeeled\b",
    r"\bchilled\b",
    r"\broom temperature\b",
    r"\bchopped\b",
    r"\bminced\b",
    r"\bfreshly\b",
    r"\bfor serving\b",
    r"\bplus extra\b",
    r"\bor substitute\b"
    # add more patterns if needed
]
NOISE_RE = re.compile("|".join(NOISE_WORDS), flags=re.IGNORECASE)

In [ ]:
from typing import List, Iterable
from collections import OrderedDict

import re
import regex            # pip install regex (better unicode support)
import math
from fractions import Fraction
from typing import List, Union, Optional
# import swifter
import unidecode

In [ ]:
# -------------------------
# 3) Unicode fraction mapping and fraction parsing
# -------------------------
UNICODE_FRACTIONS = {
    "¼": "1/4",
    "½": "1/2",
    "¾": "3/4",
    "⅐": "1/7",
    "⅑": "1/9",
    "⅒": "1/10",
    "⅓": "1/3",
    "⅔": "2/3",
    "⅕": "1/5",
    "⅖": "2/5",
    "⅗": "3/5",
    "⅘": "4/5",
    "⅙": "1/6",
    "⅚": "5/6",
    "⅛": "1/8",
    "⅜": "3/8",
    "⅝": "5/8",
    "⅞": "7/8",
}

def replace_unicode_fractions(s: str) -> str:
    for uf, ascii_frac in UNICODE_FRACTIONS.items():
        if uf in s:
            s = s.replace(uf, ascii_frac)

    return s

FRACTION_RE = regex.compile(r"(?P<mixed>\d+\s+\d+/\d+)|(?P<fraction>\d+/\d+)|(?P<decimal>\d+(?:\.\d+)?)")

def fraction_to_decimal(match_str: str) -> Optional[float]:
    """Convert a numeric string which may be mixed fraction to float."""
    s = match_str.strip()
    # Mixed like '1 1/2'
    if " " in s and "/" in s:
        parts = s.split()
        try:
            whole = float(parts[0])
            frac = float(Fraction(parts[1]))
            return whole + frac
        except Exception:
            return None
    if "/" in s:
        try:
            return float(Fraction(s))
        except Exception:
            return None
    try:
        return float(s)
    except Exception:
        return None


In [ ]:
# -------------------------
# 4) Helper utilities
# -------------------------
def safe_lower_strip(s: str) -> str:
    return unidecode.unidecode(s).strip().lower()

def normalize_units_in_text(text: str) -> str:
    """
    Replace unit variations found as standalone words with canonical names.
    e.g. '1 c sugar' -> '1 cup sugar'
    """
    def _repl(m):
        raw = m.group(1)
        key = raw.lower().rstrip(".")
        return UNIT_MAP.get(key, raw)
    return UNIT_PATTERN.sub(_repl, text)

def remove_noise(text: str) -> str:
    # remove parenthetical noise first
    text = re.sub(r"\([^)]*\)", "", text)
    # remove common noise phrases
    text = NOISE_RE.sub("", text)
    # remove dangling conjunctions left by noise removal
    text = re.sub(r"\b(and|or)\b\s*(,|$)", r"\2", text, flags=re.IGNORECASE)
    # normalize commas and whitespace
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip(" ,")
# regex to find quantity tokens at start of ingredient, e.g. "1 1/2", "", "2", "2.5", "1/2"
QUANTITY_AT_START_RE = regex.compile(r"^\s*(?P<qty>(?:\d+\s+\d+/\d+)|(?:\d+/\d+)|(?:\d+(?:\.\d+)?)|(?:[" + "".join(map(re.escape, UNICODE_FRACTIONS.keys())) + r"]))\s*(?P<rest>.*)$", flags=regex.IGNORECASE)

def extract_and_normalize_quantity(text: str):
    """
    If a leading quantity exists, convert it to a decimal string and return (quantity_as_decimal_str, rest_text).
    Otherwise return (None, original_text).
    """
    text = text.strip()
    text = replace_unicode_fractions(text)
    #print("unicde replaced text: ",text)
    m = QUANTITY_AT_START_RE.match(text)
    if not m:
        return None, text
    qty_raw = m.group("qty")
    rest = m.group("rest")
    # convert qty_raw which may include unicode (already replaced), mixed numbers, or fraction
    #decimal = fraction_to_decimal(qty_raw)
    #if decimal is None:
        #return None, text
    # Keep fraction formatting for readability if desired; here we produce decimal with sensible formatting
    # e.g., 1.5 -> "1.5", 0.5 -> "0.5"
    # But preserve integers like 2.0 -> "2"
    #if math.isclose(decimal, round(decimal)):
        #qty_str = str(int(round(decimal)))
    #else:
        #qty_str = str(round(decimal, 3)).rstrip('0').rstrip('.')  # up to 3 decimal places
    return qty_raw.strip(), rest.strip()


In [ ]:
# -------------------------
# 6) Full single-ingredient normalize function
# -------------------------
def normalize_single_ingredient(raw: str, keep_qty: bool = True) -> str:
    """
    Normalize a single ingredient line.
    Steps:
    - unicode fraction -> ascii fraction
    - remove weird chars, collapse whitespace
    - extract and normalize quantity (if present)
    - normalize units
    - remove noise phrases
    - optionally canonicalize ingredient name via fuzzy matching
    Returns a normalized ingredient string.
    """
    if not isinstance(raw, str):
        return raw
    # unify unicode, remove weird chars
    s = unidecode.unidecode(raw)
    #print("Input string  after unidecode ",s)
    s = s.replace("\u00b0", " degrees ")   # degrees sign
    s = s.replace("\xa0", " ")
    s = s.strip()
    #print("Input string to replace_unicode_fractions :",s)
    s = replace_unicode_fractions(s)
    #print("Input string after replace_unicode_fractions :",s)
    s = re.sub(r"[\u2013\u2014\u2212]", "-", s)
    #print("Input string after normalizing dashes :",s)
    s = re.sub(r"[^\S\r\n]+", " ", s)  # collapse whitespace
    #print("Input string after normalizing whitespace :",s)

    # Remove leading bullets or numbering
    s = re.sub(r"^\s*(?:[-\u2022]|(?:\d+[\.\)]))\s+", "", s)
    #print("Input string after remove leading bullets or numbering ",s)

    # extract qty if exists
    #print("Input string for extract_and_normalize_quantity ",s)
    qty, rest = extract_and_normalize_quantity(s)
    #print("qty",qty)
    #print("rest",rest)
    if qty is not None:
        s_rest = rest
    else:
        s_rest = s

    # Remove noise
    s_rest = remove_noise(s_rest).strip()

    # Normalize units (only standalone words)
    s_rest = normalize_units_in_text(s_rest)

    # normalize compound quantities like "1 cup + 1 tbsp" inside the text
    s_rest = re.sub(r"\s+\+\s+", " + ", s_rest)
    s_rest = re.sub(r"\s*\+\s*(\d+\s*(?:/\d+)?\s*)([A-Za-z]+)\b",
                    r" + \1\2", s_rest)
    # Remove extraneous punctuation (commas plus spaces)
    s_rest = re.sub(r"\s*,\s*", ", ", s_rest)
    s_rest = s_rest.strip(", ").strip()

    
    # attempt to preserve unit if exists directly after qty in the original raw string
    # naive detection: look for unit near the start of s_rest
    # check first two tokens for a unit
    first_tokens = s_rest.split()[:2]
    unit_token = None
    for t in first_tokens:
        key = t.lower().rstrip(".")
        if key in UNIT_MAP:
            unit_token = UNIT_MAP[key]
            s_rest = re.sub(rf"^{re.escape(t)}\s*", "", s_rest, flags=re.IGNORECASE)
            break
    
    # Rebuild normalized string
    parts = []
    if keep_qty and qty is not None:
        parts.append(qty)

    if unit_token:
        parts.append(unit_token)

    # Add the remaining natural phrase

    parts.append(s_rest)


    normalized = " ".join(parts).strip()
    # Fallback: if normalized empty, return cleaned s_rest
    return normalized if normalized else s_rest


In [ ]:
import ast

# -------------------------
# 7) Pipeline over a list or DataFrame column
# -------------------------
def normalize_ingredients_list(
    ingredients: Union[List[str], str],
    keep_qty: bool = True
) -> List[str]:
    """
    Normalize a list of ingredient strings 
    - Only quantity & unit normalization

    Accepts:
    - list of strings
    - string representing a list: '["1 c sugar", "2 tbsp butter"]'
    - single string (newline or comma separated)

    Returns:
    - list of normalized ingredient strings
    """

    # ---------------------------------------------------
    # 1. Detect if a string-encoded list  parse safely
    # ---------------------------------------------------
    if isinstance(ingredients, str):
        stripped = ingredients.strip()

        # Case 1: Looks like a Python list  parse it
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
                if isinstance(parsed, list):
                    ingredients = parsed
                else:
                    # fallback to splitting
                    ingredients = [s.strip() for s in re.split(r"[\n\r,;]+", ingredients) if s.strip()]
            except Exception:
                # fallback to splitting
                ingredients = [s.strip() for s in re.split(r"[\n\r,;]+", ingredients) if s.strip()]
        else:
            # Normal case: split by newline/comma/semicolon
            ingredients = [s.strip() for s in re.split(r"[\n\r,;]+", ingredients) if s.strip()]

    # ---------------------------------------------------
    # 2. If it's already a list  clean items
    # ---------------------------------------------------
    elif isinstance(ingredients, list):
        ingredients = [str(it).strip() for it in ingredients if str(it).strip()]

    else:
        return []

    # ---------------------------------------------------
    # 3. Normalize each ingredient
    # ---------------------------------------------------
    normalized_items = []
    for raw in ingredients:
        norm = normalize_single_ingredient(raw, keep_qty=keep_qty)
        if norm:
            normalized_items.append(norm)

    return normalized_items


Ingredients to Bullet Multi Line List 

In [ ]:
import re
from typing import List, Tuple, Optional

# --- lightweight normalizers ---
UNIT_TO_ABBR = {
    "x": "",
    "pound": "lb", "pounds": "lb", "lb": "lb", "lbs": "lb",
    "tablespoon": "tbsp", "tablespoons": "tbsp", "tbsp": "tbsp",
    "teaspoon": "tsp", "teaspoons": "tsp", "tsp": "tsp",
    "cup": "cup", "cups": "cups", "c": "cup", "c.": "cup", "tsp.": "tsp", "tbsp.": "tbsp", "c": "cup", "c.": "cup", "tsp.": "tsp", "tbsp.": "tbsp",
    "can": "can", "cans": "cans", "jar": "jar", "jars": "jar",
    "box": "box", "boxes": "boxes",
    "package": "pkg", "packages": "pkgs", "pkg": "pkg", "pkgs": "pkgs",
    "ounce": "oz", "ounces": "oz", "oz": "oz",
    "gram": "g", "grams": "g", "g": "g",
    "kilogram": "kg", "kilograms": "kg", "kg": "kg",
    "milliliter": "ml", "milliliters": "ml", "ml": "ml",
    "liter": "l", "liters": "l", "l": "l", "quart": "qt", "quarts": "qt", "qt": "qt",
    "clove": "clove", "cloves": "cloves",
}

SIZE_WORDS = {"small", "medium", "large"}

# matches: qty (e.g., 1, 2.5, 1/2, 6-8), optional size word, optional unit word, then item
QTY_RE = r"(?P<qty>\d+(?:\.\d+)?(?:\s+\d+/\d+)?(?:/\d+)?(?:\s*(?:-|to)\s*\d+(?:\.\d+)?(?:/\d+)?)?)"
LINE_RE = re.compile(
    rf"^\s*{QTY_RE}\s+(?:(?P<size>small|medium|large)\s+)?(?:(?P<unit>[A-Za-z]+)\s+)?(?P<item>.+?)\s*$",
    re.IGNORECASE
)

def _title_case_food(s: str) -> str:
    """Sentence-case ingredient names for consistent bullets."""
    s = s.strip()
    if not s:
        return s
    words = s.lower().split()
    if not words:
        return ""
    words[0] = words[0][:1].upper() + words[0][1:]
    return " ".join(words)

def _clean_line(line: str) -> str:
    # strip stray quotes and whitespace
    return line.strip().strip("'").strip('"').strip()

def _normalize_unit(unit: Optional[str]) -> Optional[str]:
    if not unit:
        return None
    u = unit.strip().lower().rstrip(".")
    if u == "x":
        return None
    return UNIT_TO_ABBR.get(u, u)

def _split_notes(line: str) -> Tuple[str, str]:
    """
    Split 'chicken, boiled' => ('chicken', 'boiled')
    Only splits on first comma.
    """
    parts = [p.strip() for p in line.split(",", 1)]
    main = parts[0]
    notes = parts[1] if len(parts) > 1 else ""
    return main, notes

def ingredients_to_bullets(raw: str, bullet: str = "-", names_only: bool = False) -> str:
    """
    Convert a raw multiline ingredient string into clean human-readable bullets.

    Example input line:
      '3 pound chicken, boiled'
    Output bullet:
      '- Chicken  3 lb (boiled)'

    Handles:
    - count-based items like '4 medium potatoes, diced' -> 'Potatoes  4 medium (diced)'
    - unit normalization: pounds->lb, tablespoons->tbsp, etc.
    - trailing stray quotes
    - when names_only=True, output just ingredient names
    """
    if raw is None:
        return ""

    # split into non-empty lines
    lines = [_clean_line(l) for l in str(raw).splitlines()]
    lines = [l for l in lines if l]

    bullets: List[str] = []

    for line in lines:
        if "each:" in line.lower():
            main, note = line, ""
        else:
            main, note = _split_notes(line)
        m = LINE_RE.match(main)

        if not m:
            # fallback: no qty pattern, just emit as-is
            item_name = _title_case_food(main)
            if names_only:
                bullets.append(f"{bullet} {item_name}")
            else:
                suffix = f" ({note})" if note else ""
                bullets.append(f"{bullet} {item_name}{suffix}")
            continue

        qty = m.group("qty").strip()
        size = (m.group("size") or "").strip().lower() or None
        unit = _normalize_unit(m.group("unit"))
        item = m.group("item").strip().lstrip(".").strip()
        if item.lower().startswith("of "):
            item = item[3:]
        each_flag = False
        if item.lower().startswith("each:"):
            each_flag = True
            item = item.split(":", 1)[1].strip()

        # If unit is missing, treat it as count-based (e.g. "2 eggs")
        if unit is None:
            item_name = _title_case_food(item)
            if names_only:
                bullets.append(f"{bullet} {item_name}")
                continue
            qty_str = f"{qty}{(' ' + size) if size else ''}".strip()
            if each_flag:
                qty_str = f"{qty_str} each"
            suffix = f" ({note})" if note else ""
            bullets.append(f"{bullet} {item_name} - {qty_str}{suffix}")
            continue

        # If "unit" is actually the ingredient (e.g. "4 medium potatoes"):
        # We'll detect by checking if unit is a known unit abbreviation/value.
        known_units = set(UNIT_TO_ABBR.values())
        if unit not in known_units:
            # Interpret as count-based item: qty + size + unit(word) + item(rest)
            # Example: "4 medium potatoes" -> unit="potatoes", item="" => item="potatoes"
            combined_item = f"{unit} {item}".strip()
            item_name = _title_case_food(combined_item)
            if names_only:
                bullets.append(f"{bullet} {item_name}")
                continue
            qty_str = f"{qty}{(' ' + size) if size else ''}".strip()
            if each_flag:
                qty_str = f"{qty_str} each"
            suffix = f" ({note})" if note else ""
            bullets.append(f"{bullet} {item_name} - {qty_str}{suffix}")
            continue

        # Normal unit case: "3 pound chicken" -> Chicken - 3 lb
        item_name = _title_case_food(item)
        if names_only:
            bullets.append(f"{bullet} {item_name}")
            continue
        qty_unit = f"{qty} {unit}".strip()
        if size:
            # Usually size belongs with the item when there is a unit ("1 small box")
            qty_unit = f"{qty} {size} {unit}".strip()
        if each_flag:
            qty_unit = f"{qty_unit} each"

        suffix = f" ({note})" if note else ""
        bullets.append(f"{bullet} {item_name} - {qty_unit}{suffix}")

    return "\n".join(bullets)

Ingredients as bullets 

In [ ]:
import ast

def _ingredients_to_multiline(x):
    if isinstance(x, list):
        return "\n".join(x)
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return "\n".join(parsed)
        except Exception:
            pass
        return x.replace("\\n", "\n")
    return ""
'''
if "ingredients" in df.columns:
    df["ingredients_bullets"] = df["ingredients"].map(
        lambda x: ingredients_to_bullets(_ingredients_to_multiline(x))
    )
    df["ingredients_names_only"] = df["ingredients"].map(
        lambda x: ingredients_to_bullets(_ingredients_to_multiline(x), names_only=True)
    )
'''

Title Cleaning

In [ ]:
import re
import unidecode

def normalize_title(title: str) -> str:
    if not isinstance(title, str):
        return ""

    s = unidecode.unidecode(title).strip()
    s = re.sub(r"\s+", " ", s)

    # Remove leading unmatched parentheses like "(" or "(("
    s = re.sub(r"^[\(\)]+(?=\w)", "", s).strip()

    # Normalize trailing duplicates like "))))"
    s = re.sub(r"[\(\)]+$", "", s).strip()

    # If there are more opening than closing parens, close them at the end.
    open_count = s.count("(")
    close_count = s.count(")")
    if open_count > close_count:
        s = s + (")" * (open_count - close_count))

    # Remove trailing punctuation
    s = re.sub(r"[!?.,;:]+$", "", s)

    return s.title()


In [ ]:
def normalize_directions(directions):
    if isinstance(directions, list):
        steps = [unidecode.unidecode(str(s)).strip() for s in directions if str(s).strip()]
        return " ".join(steps)

    s = unidecode.unidecode(str(directions)).strip()
    s = re.sub(r"\s+", " ", s)
    return s

In [ ]:
# ==========================================
# 5. SUSPICIOUS-RECIPE FILTERS (safe)
# ==========================================
def filter_suspicious(df):
    # Remove empty or malformed ingredients
    df = df[df["ingredients_normalized"].apply(lambda x: isinstance(x, list) and len(x) > 3)]
    df = df[df["ingredients_normalized"].apply(lambda x: len(x) <= 100)]  # upper bound

    # Remove weirdly long/short directions
    df = df[df["directions_normalized"].str.len() > 10]
    df = df[df["directions_normalized"].str.len() < 10000]

    # Token-based filtering (MOST IMPORTANT)
    df = df[(df["input_tokens"] >= 5)]
    df = df[(df["output_tokens"] >= 10)]
    df = df[(df["output_tokens"] <= 1500)]

    return df

In [ ]:
def apply_full_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    df_temp = df  # or df.copy(deep=False)

    df_temp["title_normalized"] = df_temp["title"].map(normalize_title)

    df_temp["ingredients_normalized"] = df_temp["ingredients"].map(
        lambda x: normalize_ingredients_list(x, keep_qty=True)
    )

    df_temp["directions_normalized"] = df_temp["directions"].map(
        normalize_directions
    )

    ing = df_temp["ingredients_normalized"]
    df_temp["input_tokens"] = ing.map(
        lambda x: sum(len(t.split()) for t in x) if isinstance(x, list) else 0
    )

    df_temp["ingredients_bullets"] = df["ingredients_normalized"].map(
        lambda x: ingredients_to_bullets(_ingredients_to_multiline(x))
    )
    df["ingredients_names_only"] = df["ingredients_normalized"].map(
        lambda x: ingredients_to_bullets(_ingredients_to_multiline(x), names_only=True)
    )

    df_temp["output_tokens"] = df_temp["directions_normalized"].map(
        lambda x: len(str(x).split())
    )

    return filter_suspicious(df_temp)


In [ ]:
df_filtered = df.copy()
#df_filtered = df.sample(n=200, random_state=42).copy()
df_filtered = apply_full_pipeline(df_filtered)
df_filtered.to_csv("../data/dataset_Read_V6_filtered_v3.csv", index=False)

In [ ]:
df_filtered.head()

In [ ]:
print(df_filtered['ingredients'].iloc[156])

In [ ]:
print(df_filtered['ingredients_normalized'].iloc[186])

In [ ]:
print(df_filtered['ingredients_bullets'].iloc[186])

In [ ]:
type(df_filtered['directions_normalized'].iloc[0])
#print(df_filtered['directions_normalized'].iloc[0])


### Blocked similarity clustering (MinHash LSH)


In [ ]:
from datasketch import MinHash, MinHashLSH

COMMON_ING = {
    "salt", "pepper", "water", "oil", "olive oil", "vegetable oil",
    "sugar", "flour", "butter", "milk"
}

def title_prefix_key(title_norm, k=5):
    toks = (title_norm or "").split()
    return " ".join(toks[:k])

def ingredient_signature(ingredients_norm, k=5):
    toks = [x for x in (ingredients_norm or []) if x not in COMMON_ING]
    toks.sort()
    return "|".join(toks[:k])

# Title frequency -> identify generic titles
_title_counts = df_filtered["title"].value_counts()
df_filtered["title_freq"] = df_filtered["title"].map(_title_counts).astype("int32")
df_filtered["title_is_generic"] = df_filtered["title_freq"] >= 300

# Block key
_df_title_norm = df_filtered["title_normalized"]
df_filtered["title_prefix"] = _df_title_norm.apply(lambda s: title_prefix_key(s, k=5))
df_filtered["block_key"] = df_filtered.apply(
    lambda r: r["title_prefix"] if not r["title_is_generic"]
    else ingredient_signature(r["ingredients_normalized"], k=5),
    axis=1
)

# Block size diagnostics (inspect top heavy blocks)
block_sizes = df_filtered.groupby("block_key").size().sort_values(ascending=False)
block_sizes.head(20)


In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0]*n
    def find(self, a):
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1

def minhash(tokens, num_perm=64):
    m = MinHash(num_perm=num_perm)
    for t in tokens or []:
        m.update(t.encode("utf8"))
    return m

def cluster_block(ingredients_lists, threshold=0.85, num_perm=64):
    n = len(ingredients_lists)
    uf = UnionFind(n)

    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    mhs = []

    for i, toks in enumerate(ingredients_lists):
        mh = minhash(toks, num_perm=num_perm)
        mhs.append(mh)
        lsh.insert(str(i), mh)

    for i, mh in enumerate(mhs):
        for c in lsh.query(mh):
            j = int(c)
            if j <= i:
                continue
            if mh.jaccard(mhs[j]) >= threshold:
                uf.union(i, j)

    root_to_id = {}
    local_ids = [0]*n
    next_id = 0
    for i in range(n):
        r = uf.find(i)
        if r not in root_to_id:
            root_to_id[r] = next_id
            next_id += 1
        local_ids[i] = root_to_id[r]

    return local_ids

def _subkey_from_ingredients(xs):
    if isinstance(xs, list) and xs:
        return "|".join(xs[:2])
    return ""


In [ ]:

THRESH = 0.85
NUM_PERM = 64
MAX_BLOCK = 3000

id_col = "recipe_id" if "recipe_id" in df_filtered.columns else None
if id_col is not None and not df_filtered[id_col].is_unique:
    id_col = None
if id_col is None:
    df_filtered = df_filtered.reset_index(drop=False).rename(columns={"index": "row_id"})
    id_col = "row_id"

out_rows = []
global_cluster = 0

for block_key, sub in df_filtered.groupby("block_key", sort=False):
    if len(sub) == 1:
        out_rows.append((sub[id_col].iloc[0], global_cluster))
        global_cluster += 1
        continue

    if len(sub) > MAX_BLOCK:
        sub = sub.copy()
        sub["subkey"] = sub["ingredients_normalized"].apply(_subkey_from_ingredients)
        for _, sub2 in sub.groupby("subkey", sort=False):
            local = cluster_block(sub2["ingredients_normalized"].tolist(), threshold=THRESH, num_perm=NUM_PERM)
            mapping = {}
            for lid in local:
                if lid not in mapping:
                    mapping[lid] = global_cluster
                    global_cluster += 1
            for rid, lid in zip(sub2[id_col].tolist(), local):
                out_rows.append((rid, mapping[lid]))
        continue

    local = cluster_block(sub["ingredients_normalized"].tolist(), threshold=THRESH, num_perm=NUM_PERM)
    mapping = {}
    for lid in local:
        if lid not in mapping:
            mapping[lid] = global_cluster
            global_cluster += 1
    for rid, lid in zip(sub[id_col].tolist(), local):
        out_rows.append((rid, mapping[lid]))

clusters = pd.DataFrame(out_rows, columns=[id_col, "ingredient_cluster_id"])
df_filtered = df_filtered.merge(clusters, on=id_col, how="left")

df_filtered[["title", "ingredient_cluster_id"]].head()

In [ ]:
# Self Validation

sizes = df_filtered.groupby("ingredient_cluster_id").size().sort_values(ascending=False)
print(sizes.head(20))

big_cluster = sizes.index[0]
df_filtered[df_filtered["ingredient_cluster_id"] == big_cluster][["title","NER"]].head(10)


In [ ]:
# Alternative: Show all matching items with their cluster IDs
matching_items = df_filtered[df_filtered["title"] == "Chicken Casserole"][["title", "ingredient_cluster_id"]]
print(matching_items)

In [ ]:
df_filtered.columns

In [ ]:
df_filtered[df_filtered['ingredient_cluster_id'] == 10761][['title','ingredients_normalized', 'NER']].head()

In [ ]:
title = "Chicken Casserole"

stats = (
    df_filtered[df_filtered["title"] == title]
      .groupby("ingredient_cluster_id")
      .size()
      .describe()
)

print(stats)
print("Unique clusters:", df_filtered[df_filtered["title"] == title]["ingredient_cluster_id"].nunique())


In [ ]:
cluster_sizes = df_filtered.groupby("ingredient_cluster_id").size()

print(cluster_sizes.describe(percentiles=[.5, .9, .99, .999]))

cluster_sizes.hist(bins=50, log=True)

In [ ]:
def list_length_count(list_str):
    """
    The NER column is a string representation of a list.
    This function takes a string representation of a list and returns the count of elements in the list.
    Return , We get the number of Ingredients
    """
    if isinstance(list_str, list):
        return len(list_str)
    if not isinstance(list_str, str):
        return 0
    try:
        parsed = ast.literal_eval(list_str)
        return len(parsed) if isinstance(parsed, list) else 0
    except (ValueError, SyntaxError):
        return 0

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(15,5))

ax1.plot(df_filtered['input_tokens'])
ax1.set_xlabel('Index')
ax1.set_ylabel('input_tokens')
ax1.set_title('input_tokens over Index')

ax2.plot(df_filtered['output_tokens'])
ax2.set_xlabel('Index')
ax2.set_ylabel('output_tokens')
ax2.set_title('output_tokens over Index')

plt.tight_layout()
plt.show()

In [ ]:
df_filtered.describe()

Split for the test and eval 

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def grouped_train_val_test_split(df, group_col="ingredient_cluster_id", train_size=0.90, val_size=0.05, test_size=0.05, seed=42):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-9

    gss1 = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    train_idx, temp_idx = next(gss1.split(df, groups=df[group_col]))

    df_train = df.iloc[train_idx].copy()
    df_temp = df.iloc[temp_idx].copy()

    # split temp into val/test (relative proportions)
    val_frac_of_temp = val_size / (val_size + test_size)

    gss2 = GroupShuffleSplit(n_splits=1, train_size=val_frac_of_temp, random_state=seed)
    val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp[group_col]))

    df_val = df_temp.iloc[val_idx].copy()
    df_test = df_temp.iloc[test_idx].copy()

    return df_train, df_val, df_test

train_df, val_df, test_df = grouped_train_val_test_split(df_filtered, group_col="ingredient_cluster_id", seed=42)
len(train_df), len(val_df), len(test_df)


Sanity Check 

In [ ]:
# Sizes
print("train/val/test sizes:", len(train_df), len(val_df), len(test_df))
print("total:", len(train_df) + len(val_df) + len(test_df))

# No overlap in ids
id_col = "recipe_id" if "recipe_id" in df_filtered.columns else "row_id"
train_ids = set(train_df[id_col])
val_ids = set(val_df[id_col])
test_ids = set(test_df[id_col])
print("id overlap train/val:", len(train_ids & val_ids))
print("id overlap train/test:", len(train_ids & test_ids))
print("id overlap val/test:", len(val_ids & test_ids))

# No cluster leakage
group_col = "ingredient_cluster_id"
train_groups = set(train_df[group_col])
val_groups = set(val_df[group_col])
test_groups = set(test_df[group_col])
print("cluster overlap train/val:", len(train_groups & val_groups))
print("cluster overlap train/test:", len(train_groups & test_groups))
print("cluster overlap val/test:", len(val_groups & test_groups))

# Spot-check a few clusters
sample_groups = list(train_groups)[:3]
for g in sample_groups:
    print("cluster", g)
    display(train_df[train_df[group_col] == g][["title", "ingredients_normalized"]].head(3))


In [ ]:
import ast

def _format_directions(d):
    if isinstance(d, list):
        return "\n".join(f"{i+1}. {str(step).strip()}" for i, step in enumerate(d) if str(step).strip())
    if isinstance(d, str):
        s = d.strip()
        # try to parse string-list
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return "\n".join(f"{i+1}. {str(step).strip()}" for i, step in enumerate(parsed) if str(step).strip())
            except Exception:
                pass
        return s
    return str(d).strip()


In [ ]:
import ast
import json

def export_to_jsonl(df, path="recipes_cleaned.jsonl"):
    """
    Export a cleaned recipe DataFrame to HuggingFace-style JSONL.
    Optimized for performance with proper error handling.
    """
    # Validate required columns
    required_cols = ["title_normalized", "ingredients_normalized", "directions_normalized","ingredients_bullets","NER"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    records = []
    
    
    # Use itertuples for better performance (~10-100x faster than iterrows)
    for row in df.itertuples(index=False):
        # Safely handle ingredients for input
        ingredients = getattr(row, "ingredients_bullets", None)
        if isinstance(ingredients, list):
            pass
        elif isinstance(ingredients, str):
            ingredients = [ln.strip() for ln in ingredients.splitlines() if ln.strip()]
        else:
            ingredients_norm = getattr(row, "ingredients_normalized", [])
            ingredients = ingredients_norm if isinstance(ingredients_norm, list) else []
        
        # Build input_text with f-strings for better readability
        input_text = (
            f"Title: {row.title_normalized}\n\n"
            "Ingredients:\n"
            + "\n".join(ingredients)
        )

        record = {
            "instruction": (
                "You are a culinary assistant. "
                "Write step-by-step cooking directions using the given title and ingredients. "
                "Use all relevant ingredients. "
                "Do NOT repeat the ingredient list. "
                "Use complete sentences. "
                "Use numbered steps with action verbs."
            ),
            "input": input_text,
            "output": _format_directions(row.directions_normalized)
        }

        # Optional metadata for analysis and filtering
        if hasattr(row, "recipe_id"):
            record["recipe_id"] = row.recipe_id
        if hasattr(row, "title_normalized"):
            record["title_normalized"] = row.title_normalized
        if hasattr(row, "ingredients_normalized"):
            record["ingredients_normalized"] = row.ingredients_normalized
        if hasattr(row, "ingredients_bullets"):
            record["ingredients_bullets"] = row.ingredients_bullets
        if hasattr(row, "ingredients_names_only"):
            record["ingredients_names_only"] = row.ingredients_names_only        
        if hasattr(row, "input_tokens"):
            record["input_tokens"] = row.input_tokens
        if hasattr(row, "output_tokens"):
            record["output_tokens"] = row.output_tokens
        if hasattr(row, "directions_normalized"):
            record["directions_normalized"] = row.directions_normalized

        if hasattr(row, "NER"):
            ner_val = row.NER
            if isinstance(ner_val, str):
                try:
                    ner_val = ast.literal_eval(ner_val)
                except (ValueError, SyntaxError):
                    pass
            record["ner_ingredients"] = ner_val

        records.append(record)

    # Write to file with error handling
    try:
        with open(path, "w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Exported {len(records)} records to {path}")
    except IOError as e:
        print(f"Error writing to {path}: {e}")
        raise

In [ ]:
import os

os.makedirs("../data", exist_ok=True)

export_to_jsonl(train_df, "../data/train_v6.jsonl")
export_to_jsonl(val_df, "../data/eval_v6.jsonl")
export_to_jsonl(test_df, "../data/test_v6.jsonl")


Function to import the dataset

In [ ]:
import json
import pandas as pd
import re

def import_from_jsonl(file_path="recipes_cleaned.jsonl"):
    """
    Import a HuggingFace-style JSONL file into a pandas DataFrame.
    Compatible with export_to_jsonl format.
    
    Args:
        file_path (str): Path to the JSONL file to import
        
    Returns:
        pd.DataFrame: DataFrame containing the imported data
    """
    data = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    # Parse each line as JSON and add to our data list
                    record = json.loads(line)
                    data.append(record)
                except json.JSONDecodeError as e:
                    print(f"Warning: Could not parse line: {line[:100]}... Error: {e}")
                    continue
        
        # Convert to DataFrame
        df = pd.DataFrame(data)
        return df
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error importing JSONL file: {e}")
        return pd.DataFrame()


In [ ]:
# To use the function:
df_imported = import_from_jsonl("../data/test_v6.jsonl")


In [ ]:
df_imported.head(5)

In [ ]:
df_imported['input'].iloc[1]

### Hugging Face

In [ ]:
from huggingface_hub import HfApi, login
from huggingface_hub.utils import HfHubHTTPError

def ensure_hf_login():
    """
    Ensure user is logged in to Hugging Face Hub.
    Prompts for login if not already authenticated.
    """


# Usage:
# if ensure_hf_login():
#     # Proceed with your Hugging Face operations
#     pass

In [ ]:
login()

In [ ]:
ensure_hf_login()

In [ ]:
# Full usage (train + test + eval)
push_to_hub(
    train_path="train.jsonl",
    test_path="test.jsonl",
    eval_path="eval.jsonl",
    repo_name="nijumich/recipieNLG_V2"
)

Load From Hugging Face

In [ ]:
from datasets import load_dataset

# Load the entire dataset
#dataset = load_dataset("nijumich/recipieNLG_V1")

# Or load specific splits
train_data = load_dataset("nijumich/recipieNLG_V2", split="train")
test_data = load_dataset("nijumich/recipieNLG_V2", split="test")
eval_data = load_dataset("nijumich/recipieNLG_V2", split="validation")  # or "eval" if you used that name

# Example: Print first training example
print(train_data[0])

# Convert to pandas DataFrame if needed
#import pandas as pd
#df_train = train_data.to_pandas()
#df_test = test_data.to_pandas()

Utility - Calculate Token Distribution

1. Step 1: Analyze Your Dataset (The Statistical Approach)

You should set your sequence length to cover most of your data without wasting memory on extreme outliers. If 95% of your examples are under 1,024 tokens, setting the length to 4,096 will waste 75% of your compute on empty padding.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset
import numpy as np

offline = True
if offline: 
    dataset = load_dataset("json", data_files={"train": "../data/train_v6.jsonl"})
    test_ds = dataset["train"]
    ft_test_data = test_ds.select(range(100))  # Take only first 100 rows
else :
    ft_dataset_name = "nijumich/recipieNLG_V1"   
    ft_data = load_dataset(ft_dataset_name)
    ft_train_data = ft_data["train"]
    ft_eval_data = ft_data["validation"]
    ft_test_data = ft_data["test"][:100]  # Take only first 100 rows
print(len(ft_test_data))

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
#dataset = load_dataset("nijumich/recipieNLG_V1", split="train")

def get_length(example):
    # Adjust 'text' to whatever your column name is
    input_length = len(tokenizer(example["input"])["input_ids"])
    output_length = len(tokenizer(example["output"])["input_ids"])
    return {
        "input_token_length": input_length,
        "output_token_length": output_length,
        "token_ratio": output_length / input_length
    }

lengths = dataset["train"].map(get_length, batched=False, remove_columns=dataset["train"].column_names)
input_token_counts = [x["input_token_length"] for x in lengths]
output_token_counts = [x["output_token_length"] for x in lengths]
token_ratios = [x["token_ratio"] for x in lengths]

print(f"input 90th percentile: {np.percentile(input_token_counts, 90)}")
print(f"input 95th percentile: {np.percentile(input_token_counts, 95)}")
print(f"input mean length: {np.mean(input_token_counts)}")
print(f"input max length: {max(input_token_counts)}")

print(f"output 90th percentile: {np.percentile(output_token_counts, 90)}")
print(f"output 95th percentile: {np.percentile(output_token_counts, 95)}")
print(f"output mean length: {np.mean(output_token_counts)}")
print(f"output max length: {max(output_token_counts)}")

print(f"token ratio 90th percentile: {np.percentile(token_ratios, 90)}")
print(f"token ratio 95th percentile: {np.percentile(token_ratios, 95)}")
print(f"token ratio mean: {np.mean(token_ratios)}")
print(f"token ratio max: {max(token_ratios)}")


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
train_ds = dataset["train"]

def add_gold_features(ex):
    in_len = len(tok(ex["input"])["input_ids"])
    out_len = len(tok(ex["output"])["input_ids"])
    ratio = out_len / max(in_len, 1)
    total = in_len + out_len
    return {
        "input_tok_len_gold": in_len,
        "output_tok_len_gold": out_len,
        "token_ratio_gold": ratio,
        "total_tok_len_gold": total,
        "gold_pass": (
            in_len >= 20 and
            60 <= out_len <= 800 and
            ratio <= 6 and
            total <= 1024
        ),
    }

train_scored = train_ds.map(add_gold_features)
train_gold = train_scored.filter(lambda ex: ex["gold_pass"])


In [ ]:
len(train_gold)

In [ ]:
import pandas as pd
import numpy as np
import hashlib

#df = ...  # your dataframe

# 1) Keep only gold
gold = train_gold.to_pandas().copy()

# 2) Make group_id to avoid leakage
def make_group_id(row):
    s = f"{row['title_normalized']}||{row['ingredients_normalized']}"
    return hashlib.md5(s.encode("utf-8")).hexdigest()

gold["group_id"] = gold.apply(make_group_id, axis=1)

# 3) Dedup: keep 1 example per group (pick the “best” row)
# Example heuristic: prefer mid-length outputs and reasonable ratios
gold = gold.sort_values(
    by=["token_ratio_gold", "total_tok_len_gold"],
    ascending=[True, True]
).drop_duplicates("group_id", keep="first")

# 4) Create length buckets (tune bins to your seq_len)
bins = [0, 256, 512, 768, 1024, 10**9]
labels = ["0-256", "257-512", "513-768", "769-1024", "1025+"]
gold["len_bucket"] = pd.cut(gold["total_tok_len_gold"], bins=bins, labels=labels, right=True)

# Optionally drop super long tail if you're training at seq_len=1024
gold = gold[gold["total_tok_len_gold"] <= 1024].copy()

# 5) Split by group_id (leak-free)
rng = np.random.default_rng(42)
unique_groups = gold["group_id"].unique()
rng.shuffle(unique_groups)

n = len(unique_groups)
test_groups = set(unique_groups[:int(0.01*n)])       # 1%
val_groups  = set(unique_groups[int(0.01*n):int(0.012*n)])  # 0.2%
train_groups= set(unique_groups[int(0.012*n):])

test = gold[gold["group_id"].isin(test_groups)]
val  = gold[gold["group_id"].isin(val_groups)]
train= gold[gold["group_id"].isin(train_groups)]

# 6) Stratified sampling by length bucket
def stratified_sample(df_in, n_total, bucket_col="len_bucket", seed=42):
    df_in = df_in.dropna(subset=[bucket_col]).copy()
    buckets = df_in[bucket_col].value_counts().index.tolist()
    n_per = max(1, n_total // len(buckets))
    parts = []
    for b in buckets:
        part = df_in[df_in[bucket_col] == b]
        take = min(n_per, len(part))
        parts.append(part.sample(take, random_state=seed))
    out = pd.concat(parts, ignore_index=True)
    # top up if short
    if len(out) < n_total:
        remaining = df_in.drop(out.index, errors="ignore")
        out = pd.concat([out, remaining.sample(n_total - len(out), random_state=seed)], ignore_index=True)
    return out

train_80k = stratified_sample(train, 80000)
val_3k    = stratified_sample(val, 3000)
test_10k  = stratified_sample(test, 10000)

# 7) Export to JSONL (keep only fields needed for training)
keep_cols = ['instruction', 'input', 'output', 'recipe_id', 'title_normalized',
 'ingredients_normalized', 'ingredients_bullets', 'ingredients_names_only',
  'input_tokens', 'output_tokens', 'directions_normalized', 'ner_ingredients',
   'input_tok_len_gold', 'output_tok_len_gold', 'token_ratio_gold', 'total_tok_len_gold']

train_80k[keep_cols].to_json("train_gold_80k.jsonl", orient="records", lines=True, force_ascii=False)
val_3k[keep_cols].to_json("val_3k.jsonl", orient="records", lines=True, force_ascii=False)
test_10k[keep_cols].to_json("test_10k.jsonl", orient="records", lines=True, force_ascii=False)

In [ ]:
train_gold


## IMPORTANT : NOT USED

In [ ]:
export_to_jsonl(train_gold, "../data/train_v6_gold.jsonl")

In [ ]:
# -------------------------
# Parallel Gold filter track (tokenizer-based)
# -------------------------
from transformers import AutoTokenizer
import numpy as np
import pandas as pd

GOLD_TOKENIZER_NAME = "Qwen/Qwen2.5-7B-Instruct"
_gold_tokenizer = AutoTokenizer.from_pretrained(GOLD_TOKENIZER_NAME)

def _gold_build_input_text(row):
    title = str(row.get("title_normalized", "") or "")
    ingredients = row.get("ingredients_bullets", "")

    if isinstance(ingredients, list):
        ing_text = "\n".join(str(x) for x in ingredients if str(x).strip())
    else:
        ing_text = str(ingredients or "")

    return f"Title: {title}\n\nIngredients:\n{ing_text}"


def _gold_build_output_text(row):
    directions = row.get("directions_normalized", "")
    if isinstance(directions, list):
        return "\n".join(str(x) for x in directions if str(x).strip())
    return str(directions or "")


def add_gold_token_features(df):
    out = df.copy()

    input_texts = out.apply(_gold_build_input_text, axis=1)
    output_texts = out.apply(_gold_build_output_text, axis=1)

    out["input_tok_len_gold"] = input_texts.map(lambda s: int(len(_gold_tokenizer(s)["input_ids"])))
    out["output_tok_len_gold"] = output_texts.map(lambda s: int(len(_gold_tokenizer(s)["input_ids"])))
    out["token_ratio_gold"] = out["output_tok_len_gold"] / out["input_tok_len_gold"].clip(lower=1)
    out["total_tok_len_gold"] = out["input_tok_len_gold"] + out["output_tok_len_gold"]
    return out


def apply_gold_filters(df):
    before = len(df)

    m_input = df["input_tok_len_gold"] >= 20
    m_output_min = df["output_tok_len_gold"] >= 60
    m_output_max = df["output_tok_len_gold"] <= 800
    m_ratio = df["token_ratio_gold"] <= 6
    m_total = df["total_tok_len_gold"] <= 1024

    mask = m_input & m_output_min & m_output_max & m_ratio & m_total

    out = df.copy()
    out["gold_pass"] = mask
    df_gold = out.loc[mask].copy()

    stats = {
        "before": int(before),
        "after": int(len(df_gold)),
        "retention_pct": float((len(df_gold) / max(before, 1)) * 100.0),
        "dropped_input_lt_20": int((~m_input).sum()),
        "dropped_output_lt_60": int((~m_output_min).sum()),
        "dropped_output_gt_800": int((~m_output_max).sum()),
        "dropped_ratio_gt_6": int((~m_ratio).sum()),
        "dropped_total_gt_1024": int((~m_total).sum()),
    }
    return df_gold, out, stats


df_gold_base = add_gold_token_features(df_filtered)
df_gold, df_gold_scored, gold_stats = apply_gold_filters(df_gold_base)

# Persist parallel Gold artifact (baseline outputs remain unchanged).
df_gold.to_csv("../data/dataset_Read_V6_gold.csv", index=False)
print("Saved Gold CSV to ../data/dataset_Read_V6_gold.csv")


In [ ]:
# Gold filter impact report
print("Gold filter stats:")
for k, v in gold_stats.items():
    print(f"{k}: {v}")

if len(df_gold) > 0:
    print("\nGold percentile sanity checks:")
    print(f"input_tok_len_gold p90: {np.percentile(df_gold['input_tok_len_gold'], 90)}")
    print(f"input_tok_len_gold p95: {np.percentile(df_gold['input_tok_len_gold'], 95)}")
    print(f"output_tok_len_gold p90: {np.percentile(df_gold['output_tok_len_gold'], 90)}")
    print(f"output_tok_len_gold p95: {np.percentile(df_gold['output_tok_len_gold'], 95)}")
    print(f"token_ratio_gold p90: {np.percentile(df_gold['token_ratio_gold'], 90)}")
    print(f"token_ratio_gold p95: {np.percentile(df_gold['token_ratio_gold'], 95)}")
    print(f"total_tok_len_gold p90: {np.percentile(df_gold['total_tok_len_gold'], 90)}")
    print(f"total_tok_len_gold p95: {np.percentile(df_gold['total_tok_len_gold'], 95)}")
else:
    print("No rows passed Gold filtering.")


In [ ]:
# Parallel Gold split (dual-track)
gold_train_df, gold_val_df, gold_test_df = grouped_train_val_test_split(
    df_gold,
    group_col="ingredient_cluster_id",
    seed=42,
)

print("gold train/val/test sizes:", len(gold_train_df), len(gold_val_df), len(gold_test_df))
print("gold total:", len(gold_train_df) + len(gold_val_df) + len(gold_test_df))

id_col = "recipe_id" if "recipe_id" in df_gold.columns else "row_id"
train_ids = set(gold_train_df[id_col])
val_ids = set(gold_val_df[id_col])
test_ids = set(gold_test_df[id_col])
print("gold id overlap train/val:", len(train_ids & val_ids))
print("gold id overlap train/test:", len(train_ids & test_ids))
print("gold id overlap val/test:", len(val_ids & test_ids))

train_groups = set(gold_train_df["ingredient_cluster_id"])
val_groups = set(gold_val_df["ingredient_cluster_id"])
test_groups = set(gold_test_df["ingredient_cluster_id"])
print("gold cluster overlap train/val:", len(train_groups & val_groups))
print("gold cluster overlap train/test:", len(train_groups & test_groups))
print("gold cluster overlap val/test:", len(val_groups & test_groups))


In [ ]:
# Parallel Gold exports (_gold suffix)
export_to_jsonl(gold_train_df, "../data/train_v6_gold.jsonl")
export_to_jsonl(gold_val_df, "../data/eval_v6_gold.jsonl")
export_to_jsonl(gold_test_df, "../data/test_v6_gold.jsonl")
